# Person 1 / Phase 1 — auditable Colab preparation

Run top to bottom in a **fresh GPU runtime**. This notebook collects context-only
inventory, source/API metadata and CPU predictions. It contains **no model load,
model forward, attention benchmark, OOM attempt or automatic GPU-package change**.

All shell/Python worker commands are visible in cells. `audit_run` saves each
command, timestamp, return code and complete stdout/stderr; notebook output also
shows a bounded preview. Export both the executed notebook and the evidence ZIP.
The committed notebook starts unexecuted: no remote execution results are implied.

The user-reported 18 October snapshot uses Python 3.13, torch 2.11/cu130 and
Transformers 5.18, unlike the older proposed stack. Do not silently substitute
stacks or adopt a math attention fallback. Runtime reconciliation and independent
review still precede any formal prediction/model measurement.


In [ ]:
import os, sys, json, hashlib, subprocess, uuid, shutil
from datetime import datetime, timezone
from pathlib import Path

# Keep the notebook itself CPU-only. A previous GPU-owning cell requires restart.
existing_torch = sys.modules.get("torch")
if existing_torch is not None and existing_torch.cuda.is_initialized():
    raise RuntimeError("Restart the runtime first: the notebook must not own a CUDA context.")

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:8]
RUN_ROOT = Path(os.environ.get("KVCACHE_NOTEBOOK_RUNS_ROOT", "/content/kvcache-evidence"))
RUN_DIR = RUN_ROOT / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=False)
JOURNAL = RUN_DIR / "commands.jsonl"

def audit_event(event):
    with JOURNAL.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(event, allow_nan=False) + "\n")
        handle.flush()
        os.fsync(handle.fileno())

def audit_run(label, command, *, cwd=None, env=None, timeout=300):
    if not label or any(c not in "abcdefghijklmnopqrstuvwxyz0123456789-" for c in label):
        raise ValueError("Use a simple unique command label")
    out_path = RUN_DIR / (label + ".stdout.txt")
    err_path = RUN_DIR / (label + ".stderr.txt")
    if out_path.exists() or err_path.exists():
        raise FileExistsError("Use a new label or a new run; prior outputs are preserved")
    audit_event({"event":"command_started", "label":label, "argv":command,
                 "cwd":str(cwd) if cwd else None,
                 "timestamp_utc":datetime.now(timezone.utc).isoformat(),
                 "allocator_env":{key:(env or os.environ).get(key) for key in
                     ("PYTORCH_CUDA_ALLOC_CONF", "PYTORCH_ALLOC_CONF")}})
    try:
        completed = subprocess.run(command, cwd=cwd, env=env, timeout=timeout,
                                   text=True, encoding="utf-8", errors="replace",
                                   capture_output=True, check=False)
    except Exception as error:
        err_path.write_text(repr(error) + "\n", encoding="utf-8")
        audit_event({"event":"command_failed_to_complete", "label":label,
                     "error":repr(error), "timestamp_utc":datetime.now(timezone.utc).isoformat()})
        raise
    out_path.write_text(completed.stdout, encoding="utf-8")
    err_path.write_text(completed.stderr, encoding="utf-8")
    audit_event({"event":"command_finished", "label":label, "returncode":completed.returncode,
                 "timestamp_utc":datetime.now(timezone.utc).isoformat(),
                 "stdout_sha256":hashlib.sha256(out_path.read_bytes()).hexdigest(),
                 "stderr_sha256":hashlib.sha256(err_path.read_bytes()).hexdigest()})
    print("COMMAND:", command)
    print("RETURN CODE:", completed.returncode)
    print(completed.stdout[:5000])
    if completed.stderr:
        print(completed.stderr[:5000])
    if completed.returncode:
        raise RuntimeError(f"{label} failed; full output is preserved in {RUN_DIR}")
    return completed

print("Evidence directory:", RUN_DIR)
print("Local /content storage is ephemeral. Export the evidence ZIP before disconnecting.")


## Repository provenance

`main` is resolved to an exact commit and recorded before calculations. For a
replay, replace `SOURCE_REF` with that recorded SHA. This clones a separate
checkout and does not reset an existing notebook/workspace. No GitHub credential
or push command is used in the runtime.


In [ ]:
REPO_URL = "https://github.com/DevanshSharma351/KV-Cache-Scaling.git"
SOURCE_REF = "main"  # Replace with the recorded SHA for an exact replay.
REPO = RUN_DIR / "source"
audit_run("git-clone", ["git", "clone", "--no-checkout", REPO_URL, str(REPO)])
resolved = audit_run("git-resolve", ["git", "rev-parse", SOURCE_REF], cwd=REPO).stdout.strip()
audit_run("git-checkout", ["git", "checkout", "--detach", resolved], cwd=REPO)
source_record = {"repository":REPO_URL, "requested_ref":SOURCE_REF, "resolved_commit":resolved}
(RUN_DIR / "source.json").write_text(json.dumps(source_record, indent=2) + "\n")
CPU_ENV = dict(os.environ, PYTHONPATH=str(REPO / "src"))
print(json.dumps(source_record, indent=2))


## CPU checks — no GPU dependency installation

The analytical package uses the standard library and is imported from the exact
checkout through `PYTHONPATH`. Keep the installed GPU stack unchanged while
collecting evidence. Any later approved stack change needs explicit install
cells, a runtime restart, a fresh inventory and a new review before preregistration.


In [ ]:
audit_run("cpu-tests", [sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"],
          cwd=REPO, env=CPU_ENV)
audit_run("package-freeze", [sys.executable, "-m", "pip", "freeze", "--all"])
audit_run("driver-before-context", ["nvidia-smi"])


## Context-only inventory in an isolated worker

The worker briefly allocates one element, frees it, records exact CUDA counters
and exits. The parent retains no CUDA context. Driver processes are recorded
while the worker's context is alive. No model class is instantiated. This fresh
observation may differ from the earlier user-reported snapshot and must keep its
own timestamp/identity; do not splice counters across observations.


In [ ]:
context_script = r"""
import json, os, sys, subprocess
from datetime import datetime, timezone
from importlib.metadata import version, PackageNotFoundError
import torch

def installed(name):
    try:
        return version(name)
    except PackageNotFoundError:
        return None

if torch.cuda.is_initialized():
    raise RuntimeError("Expected a new interpreter without an initialized CUDA context")
torch.cuda.init()
temporary = torch.empty(1, device="cuda:0")
torch.cuda.synchronize()
del temporary
torch.cuda.empty_cache()
torch.cuda.synchronize()
free_bytes, total_bytes = torch.cuda.mem_get_info(0)
snapshot = {
    "timestamp_utc":datetime.now(timezone.utc).isoformat(),
    "python":sys.version, "torch":torch.__version__,
    "torch_cuda_runtime":torch.version.cuda,
    "transformers":installed("transformers"),
    "gpu_name":torch.cuda.get_device_name(0),
    "gpu_identity_csv":subprocess.check_output(
        ["nvidia-smi", "--query-gpu=uuid,name,driver_version", "--format=csv,noheader"], text=True).strip(),
    "cuda_free_bytes":free_bytes, "cuda_total_bytes":total_bytes,
    "allocated_bytes":torch.cuda.memory_allocated(0),
    "reserved_bytes":torch.cuda.memory_reserved(0),
    "allocator_env":{key:os.environ.get(key) for key in
                     ("PYTORCH_CUDA_ALLOC_CONF", "PYTORCH_ALLOC_CONF")},
    "current_pid":os.getpid(),
    "gpu_processes":subprocess.check_output(
        ["nvidia-smi", "--query-compute-apps=pid,process_name,used_gpu_memory", "--format=csv,noheader"], text=True).strip(),
    "compiled_cuda_architectures":torch.cuda.get_arch_list(),
    "cuda_visible_devices":os.environ.get("CUDA_VISIBLE_DEVICES"),
    "collection_kind":"isolated context-only worker; no model APIs invoked"
}
print(json.dumps(snapshot, indent=2))
"""
collected = audit_run("context-inventory", [sys.executable, "-c", context_script], timeout=180)
INVENTORY_PATH = RUN_DIR / "context-inventory.json"
INVENTORY_PATH.write_text(json.dumps(json.loads(collected.stdout), indent=2) + "\n")
audit_run("driver-after-worker-exit", ["nvidia-smi"])


## Installed API/source evidence — inspection only

Version numbers alone cannot establish efficient SDPA compatibility. Record
forward/cache signatures and hashes of the installed source files. A source flag
indicates a compatibility question, not a successful kernel test. Import failures
are preserved as errors. No model, attention tensor or weights are created here.

Transformers 5.18's adapter can request native GQA for an unmasked prefill;
PyTorch 2.11 documentation lists CUDA native GQA for flash/math kernels. Our
forced efficient backend therefore needs explicit reconciliation. Do not run
model memory probes to discover this before preregistration.


In [ ]:
inspection_script = r"""
import hashlib, inspect, json
from pathlib import Path
report = {"inspection_only":True, "model_instantiated":False}
try:
    import torch
    import transformers
    from transformers.models.qwen2.modeling_qwen2 import Qwen2ForCausalLM
    from transformers.cache_utils import DynamicCache, StaticCache
    import transformers.integrations.sdpa_attention as sdpa
    objects = {"Qwen2ForCausalLM.forward":Qwen2ForCausalLM.forward,
               "DynamicCache":DynamicCache, "StaticCache":StaticCache,
               "sdpa_attention_forward":sdpa.sdpa_attention_forward}
    report["torch"] = torch.__version__
    report["transformers"] = transformers.__version__
    report["signatures"] = {name:str(inspect.signature(obj)) for name,obj in objects.items()}
    report["source_files"] = {}
    for obj in objects.values():
        filename = inspect.getsourcefile(obj)
        if filename:
            path = Path(filename)
            report["source_files"][path.name] = hashlib.sha256(path.read_bytes()).hexdigest()
    adapter_source = inspect.getsource(sdpa.sdpa_attention_forward)
    report["adapter_contains_enable_gqa"] = "enable_gqa" in adapter_source
    report["kernel_execution_tested"] = False
except Exception as error:
    report["inspection_error"] = repr(error)
print(json.dumps(report, indent=2))
"""
audit_run("installed-api-inspection", [sys.executable, "-c", inspection_script], timeout=180)


## Conditional calculation from exact free bytes

The calculation below uses the original shape scenarios with the fresh free-byte
budget. It does **not** freeze a prediction for an unreviewed runtime. Context is
already deducted from free memory, so no second context deduction is made.
The source report separately exposes any other GPU processes and stack mismatch.


In [ ]:
prediction_script = r"""
import hashlib, json, sys
from pathlib import Path
from kv_cache_scaling.environment import context_prediction
from kv_cache_scaling.model_math import Architecture
source = Path(sys.argv[1])
config = Path("configs/models/qwen2.5-1.5b.json")
arch = Architecture.from_config(json.loads(config.read_text()))
report = context_prediction(json.loads(source.read_text()), arch)
report["snapshot_sha256"] = hashlib.sha256(source.read_bytes()).hexdigest()
report["config_sha256"] = hashlib.sha256(config.read_bytes()).hexdigest()
Path(sys.argv[2]).write_text(json.dumps(report, indent=2) + "\n")
print(json.dumps(report, indent=2))
"""
REPORT_PATH = RUN_DIR / "conditional-prediction.json"
audit_run("conditional-prediction", [sys.executable, "-c", prediction_script,
          str(INVENTORY_PATH), str(REPORT_PATH)], cwd=REPO, env=CPU_ENV)
report = json.loads(REPORT_PATH.read_text())
assert report["gpu_measurements_authorized"] is False
print("MODEL MEASUREMENT GATE: CLOSED — reconcile runtime and obtain review/receipt first.")


## Reproduce the generic analytical draft

These four budgets remain hypothetical examples. The fresh observed budget is
handled separately above. Existing tracked drafts are never overwritten.


In [ ]:
audit_run("generic-prediction-draft", [sys.executable, "-m", "kv_cache_scaling",
          "prepare-prereg", "--output", str(RUN_DIR / "generic-theory-draft")], cwd=REPO, env=CPU_ENV)


## Export the execution trail

This cell packages completed command logs, inventory, API metadata, source SHA,
conditional calculations and generic draft. The source checkout is omitted from
the ZIP: the recorded Git SHA identifies it. Save this executed notebook with
outputs through Colab's normal **File → Download → Download .ipynb** action too.
The notebook's remote outputs have not been manufactured in the repository.


In [ ]:
import zipfile
files = [path for path in RUN_DIR.rglob("*")
         if path.is_file() and REPO not in path.parents and path.name != "evidence-manifest.json"]
manifest = {"resolved_source_commit":resolved, "files_sha256":{
    str(path.relative_to(RUN_DIR)):hashlib.sha256(path.read_bytes()).hexdigest()
    for path in sorted(files)}}
manifest_path = RUN_DIR / "evidence-manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2) + "\n")
archive = RUN_DIR.parent / (RUN_DIR.name + ".zip")
with zipfile.ZipFile(archive, "x", compression=zipfile.ZIP_DEFLATED) as bundle:
    for path in sorted(files + [manifest_path]):
        bundle.write(path, arcname=str(path.relative_to(RUN_DIR)))
print("Evidence ZIP:", archive)
print("Archive SHA-256:", hashlib.sha256(archive.read_bytes()).hexdigest())
print("Export the executed .ipynb as well; keep both until reviewed results are committed.")


In [ ]:
from google.colab import files
files.download(str(archive))


## Next gate

Review the archive and runtime mismatch before changing package pins or freezing
anything. A compatible, reviewed protocol and actual GPU-specific prediction
must be pushed, followed by a separate receipt, before model loading/forwards.
This notebook deliberately stops at preparation. Future model/sweep commands
will be added as named cells after their implementations and gates exist.
